# NLP Assignment – Test Set 1 (NERC)

This notebook solves NER-test.set using two Named Entity Recognition and Classification (NERC) systems:

1. **NLTK `VADER`**
2. **Rule-based Gazetteer + POS Heuristics**



In [23]:
import re
import pandas as pd
import nltk
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
import nltk
nltk.download("vader_lexicon")
from nltk.sentiment import SentimentIntensityAnalyzer


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/sentaj/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


## 1. Load the Test Set

The dataset contains:
- sentence id
- token id
- token
- gold BIO label


In [24]:
# Load sentiment dataset

df = pd.read_csv("unified_sentiment_topic_sentences.csv")

# Sort the data by keeping only sentences with at least 15 words
df = df[df["text"].str.split().str.len() >= 15].reset_index(drop=True)

df = df.sample(n=5000, random_state=42).reset_index(drop=True)
print(df.head())
print(df.shape)
display(df.head())


                                                text sentiment  topic
0  bottom line : " deuce " doesn't offer anything...  negative  movie
1  I am a huge fan of Kyle Mills - usually.This t...  negative   book
2  her latest relationship , with a testosterone-...  negative  movie
3  Some of Carl Jung's concepts have yet to be ex...  positive   book
4  where the fugitive surprises the most is the s...  positive  movie
(5000, 3)


,text,sentiment,topic
0,"bottom line : "" deuce "" doesn't offer anything...",negative,movie
1,I am a huge fan of Kyle Mills - usually.This t...,negative,book
2,"her latest relationship , with a testosterone-...",negative,movie
3,Some of Carl Jung's concepts have yet to be ex...,positive,book
4,where the fugitive surprises the most is the s...,positive,movie


In [25]:
# Convert sentiment dataset into the format expected by the rest of the notebook

sentences = [str(text).split() for text in df["text"]]

# Use sentiment labels as the gold labels
gold_tags_per_sent = [[label] * len(tokens)
                      for tokens, label in zip(sentences, df["sentiment"])]

print("Number of sentences:", len(sentences))
print("Example sentence:")
print(sentences[0])
print("Example labels:")
print(gold_tags_per_sent[0][:10])


Number of sentences: 5000
Example sentence:
['bottom', 'line', ':', '"', 'deuce', '"', "doesn't", 'offer', 'anything', 'new', 'but', 'it', 'is', 'intermittently', 'funny', '.']
Example labels:
['negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative', 'negative']


# 2. System 1 — NLTK `ne_chunk`

This system uses:
- tokenization
- POS tagging
- statistical named entity chunking

The labels are converted to CoNLL BIO format.


In [26]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def run_nltk(sentences):

    predictions = []

    for toks in sentences:

        text = " ".join(toks)

        score = sia.polarity_scores(text)["compound"]

        if score >= 0.05:
            label = "positive"
        elif score <= -0.05:
            label = "negative"
        else:
            label = "neutral"

        predictions.append([label] * len(toks))

    return predictions

# 3. System 2 — Rule-Based System

This system uses:
- Gazetteers
- POS heuristics
- Capitalization patterns


In [27]:
POSITIVE_WORDS = {
    "good", "great", "excellent", "amazing", "awesome",
    "love", "loved", "fun", "enjoyed", "best",
    "wonderful", "fantastic", "interesting", "powerful"
}

NEGATIVE_WORDS = {
    "bad", "terrible", "awful", "boring", "worst",
    "hate", "hated", "poor", "disappointing",
    "stupid", "dull", "predictable", "waste"
}


def rule_based_sentiment(tokens):

    score = 0

    for token in tokens:
        token = token.lower()

        if token in POSITIVE_WORDS:
            score += 1

        elif token in NEGATIVE_WORDS:
            score -= 1

    if score > 0:
        label = "positive"
    elif score < 0:
        label = "negative"
    else:
        label = "neutral"

    return [label] * len(tokens)


def run_rulebased(sentences):

    return [rule_based_sentiment(tokens) for tokens in sentences]

# 4. Run Both Systems

In [28]:
nltk_preds = run_nltk(sentences)

rule_preds = run_rulebased(sentences)

print("Predictions completed.")

Predictions completed.


# 5. Evaluation

Evaluation metrics:
- Precision
- Recall
- F1-score


In [30]:
def flatten(list_of_lists):
    return [item for sublist in list_of_lists for item in sublist]


def evaluate(name, gold, pred):

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    # Flatten BIO sequences
    gold_flat = flatten(gold)
    pred_flat = flatten(pred)

    # Encode labels
    le = LabelEncoder()

    all_labels = gold_flat + pred_flat

    le.fit(all_labels)

    gold_encoded = le.transform(gold_flat)
    pred_encoded = le.transform(pred_flat)

    # Classification report
    print(
        classification_report(
            gold_encoded,
            pred_encoded,
            target_names=le.classes_,
            zero_division=0
        )
    )

    # Overall scores
    p, r, f, _ = precision_recall_fscore_support(
        gold_encoded,
        pred_encoded,
        average="micro",
        zero_division=0
    )

    print(f"Micro Precision: {p:.3f}")
    print(f"Micro Recall:    {r:.3f}")
    print(f"Micro F1-score:  {f:.3f}")


evaluate("System 1 — NLTK VADER", gold_tags_per_sent, nltk_preds)

evaluate("System 2 — Rule-based", gold_tags_per_sent, rule_preds)


System 1 — VADER
              precision    recall  f1-score   support

    negative       0.43      0.39      0.41     40085
     neutral       0.08      0.17      0.11      9635
    positive       0.68      0.62      0.65     81799

    accuracy                           0.52    131519
   macro avg       0.40      0.39      0.39    131519
weighted avg       0.56      0.52      0.53    131519

Micro Precision: 0.516
Micro Recall:    0.516
Micro F1-score:  0.516

System 2 — Rule-based
              precision    recall  f1-score   support

    negative       0.64      0.07      0.13     40085
     neutral       0.07      0.77      0.13      9635
    positive       0.72      0.20      0.31     81799

    accuracy                           0.20    131519
   macro avg       0.48      0.35      0.19    131519
weighted avg       0.65      0.20      0.24    131519

Micro Precision: 0.202
Micro Recall:    0.202
Micro F1-score:  0.202


# 6. Per-Sentence Comparison

In [33]:
def sentence_label(tags):
    return tags[0] if len(tags) > 0 else "unknown"


for sid, (toks, gold, nltk_p, rule_p) in enumerate(
    zip(sentences, gold_tags_per_sent, nltk_preds, rule_preds)
):

    print("\n" + "-" * 70)
    print(f"Sentence {sid}")
    print("-" * 70)

    print(" ".join(toks))

    print("\nGold:")
    print(sentence_label(gold))

    print("\nNLTK:")
    print(sentence_label(nltk_p))

    print("\nRule-based:")
    print(sentence_label(rule_p))


----------------------------------------------------------------------
Sentence 0
----------------------------------------------------------------------
bottom line : " deuce " doesn't offer anything new but it is intermittently funny .

Gold:
negative

NLTK:
positive

Rule-based:
neutral

----------------------------------------------------------------------
Sentence 1
----------------------------------------------------------------------
I am a huge fan of Kyle Mills - usually.This thriller is anything but thrilling.

Gold:
negative

NLTK:
positive

Rule-based:
neutral

----------------------------------------------------------------------
Sentence 2
----------------------------------------------------------------------
her latest relationship , with a testosterone-overdosed drug-dealer , came to an abrupt end when she refused to wear a beeper .

Gold:
negative

NLTK:
negative

Rule-based:
neutral

----------------------------------------------------------------------
Sentence 3
---

# Now with the test set:

In [35]:
test_df = pd.read_csv("Sentiment-topic-test.tsv", sep="\t")

# Optional: apply the same 15-word filter
test_df = test_df[
    test_df["text"].str.split().str.len() >= 15
].reset_index(drop=True)

test_sentences = [str(text).split() for text in test_df["text"]]

test_gold = [
    [label] * len(tokens)
    for tokens, label in zip(test_sentences, test_df["sentiment"])
]

In [36]:
nltk_test_preds = run_nltk(test_sentences)
rule_test_preds = run_rulebased(test_sentences)

In [37]:
evaluate(
    "NLTK on unseen test set",
    test_gold,
    nltk_test_preds
)

evaluate(
    "Rule-based on unseen test set",
    test_gold,
    rule_test_preds
)


NLTK on unseen test set
              precision    recall  f1-score   support

    negative       1.00      0.44      0.62        36
     neutral       1.00      0.29      0.45        66
    positive       0.49      1.00      0.66        65

    accuracy                           0.60       167
   macro avg       0.83      0.58      0.57       167
weighted avg       0.80      0.60      0.57       167

Micro Precision: 0.599
Micro Recall:    0.599
Micro F1-score:  0.599

Rule-based on unseen test set
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        36
     neutral       0.45      1.00      0.62        66
    positive       1.00      0.29      0.45        65

    accuracy                           0.51       167
   macro avg       0.48      0.43      0.36       167
weighted avg       0.57      0.51      0.42       167

Micro Precision: 0.509
Micro Recall:    0.509
Micro F1-score:  0.509


In [38]:
for toks, gold, nltk_p, rule_p in zip(
    test_sentences,
    test_gold,
    nltk_test_preds,
    rule_test_preds
):

    print("\nSentence:")
    print(" ".join(toks))

    print("Gold:      ", gold[0])
    print("NLTK:      ", nltk_p[0])
    print("Rule-based:", rule_p[0])


Sentence:
It took eight years for Warner Brothers to recover from the disaster that was this movie.
Gold:       negative
NLTK:       negative
Rule-based: neutral

Sentence:
All the New York University students love this diner in Soho so it makes for a fun young atmosphere.
Gold:       positive
NLTK:       positive
Rule-based: positive

Sentence:
This Italian place is really trendy but they have forgotten about the most important part of a restaurant, the food.
Gold:       negative
NLTK:       positive
Rule-based: neutral

Sentence:
In conclusion, my review of this book would be: I like Jane Austen and understand why she is famous.
Gold:       positive
NLTK:       positive
Rule-based: neutral

Sentence:
The story of this movie is focused on Carl Brashear played by Cuba Gooding Jr. who wants to be the first African American deep sea diver in the Navy.
Gold:       neutral
NLTK:       positive
Rule-based: neutral

Sentence:
Chris O'Donnell stated that while filming for this movie, he felt